# Verify batched `generate_img2img`

Confirms that the vectorized pipeline (B > 1) produces pixel-exact outputs
compared to sequential B=1 calls under three conditions:

1. **B=1 regression** — single image + 1-D siglip, no object mask.
2. **B=4 identical inputs** — same image and embedding replicated; all four
   outputs must be identical to each other and to the B=1 result.
3. **B=4 distinct inputs** — four different stimuli; each batched output must
   match its corresponding B=1 call pixel-for-pixel.
4. **B=4 with per-sample `object_mask`** — masks with different bbox sizes;
   checks the per-sample loop in `IPAFluxAttnProcessor` against B=1 calls.
5. **Guard clauses** — `use_rf_inversion=True` with B > 1 raises `NotImplementedError`.

In [ ]:
import sys
import torch
import numpy as np
from PIL import Image

sys.path.insert(0, '..')
from config_const import (
    SEED, HVM_STIM_DIR, HVM_N_VAR, HVM_CATEGORIES,
    HVM_SIGLIP_EMBEDDINGS_PATH,
)
from data_utils.hvm_loader import _category_stratified_split
from generation.flux_instantx import load_pipeline, generate_img2img
from generation.aperture import load_hvm_packed_aperture_mask, build_object_region_mask
from get_device import get_device

DEVICE     = get_device()
IMAGE_SIZE = 512
STRENGTH   = 0.65
STEPS      = 20
SEED_GEN   = 7

pipe, image_proj = load_pipeline(device=DEVICE, default_scale=1.0)
aperture = load_hvm_packed_aperture_mask(image_size=IMAGE_SIZE, device='cpu', dtype=torch.bfloat16)

_, _, test_idx = _category_stratified_split(SEED)
test_idx = test_idx[:4]
siglip_gt = torch.load(HVM_SIGLIP_EMBEDDINGS_PATH, weights_only=True)

def load_stim(i):
    cat = HVM_CATEGORIES[i // HVM_N_VAR]
    var = i % HVM_N_VAR
    return Image.open(HVM_STIM_DIR / cat / f'{var:02d}.png').convert('RGB').resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)

originals = [load_stim(int(i)) for i in test_idx]
embeddings = siglip_gt[test_idx]   # (4, 1152)
print(f'loaded {len(originals)} stimuli, embeddings {tuple(embeddings.shape)}')

In [ ]:
def shared_kwargs(seed=SEED_GEN):
    return dict(
        strength=STRENGTH, prompt='',
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        num_inference_steps=STEPS, guidance_scale=3.5,
        ip_adapter_scale=1.0, aperture_mask=aperture,
        seed=seed, show_progress=False,
    )

def imgs_equal(a, b):
    """Return True when two PIL images are pixel-exact."""
    return np.array_equal(np.array(a), np.array(b))

def max_abs_diff(a, b):
    return int(np.abs(np.array(a).astype(int) - np.array(b).astype(int)).max())

## Test 1 — B=1 regression

Single PIL image + 1-D siglip. Confirms B=1 return type is `Image.Image`
and the result is pixel-exact when called twice with the same seed.

In [ ]:
out_a = generate_img2img(pipe, image_proj, originals[0], embeddings[0], **shared_kwargs())
out_b = generate_img2img(pipe, image_proj, originals[0], embeddings[0], **shared_kwargs())

assert isinstance(out_a, Image.Image), f'expected Image.Image, got {type(out_a)}'
assert imgs_equal(out_a, out_b), f'B=1 not deterministic, max diff={max_abs_diff(out_a, out_b)}'
print('Test 1 PASSED: B=1 returns Image.Image and is deterministic')

## Test 2 — B=4 identical inputs

All four slots receive the same image and embedding. Every output must be
pixel-exact with each other and with the B=1 result.

In [ ]:
emb_batch = embeddings[0:1].expand(4, -1)   # (4, 1152) — same embedding
imgs_batch = [originals[0]] * 4

out_batch = generate_img2img(pipe, image_proj, imgs_batch, emb_batch, **shared_kwargs())
assert isinstance(out_batch, list) and len(out_batch) == 4, \
    f'expected list[Image.Image] len 4, got {type(out_batch)}'

for k, out_k in enumerate(out_batch):
    assert imgs_equal(out_k, out_a), \
        f'batch[{k}] differs from B=1 result, max diff={max_abs_diff(out_k, out_a)}'

print('Test 2 PASSED: B=4 identical inputs all match the B=1 result pixel-exactly')

## Test 3 — B=4 distinct inputs

Four different stimuli run as a single batch. Each output must match the
corresponding B=1 call pixel-for-pixel.

In [ ]:
b1_results = [
    generate_img2img(pipe, image_proj, originals[k], embeddings[k], **shared_kwargs())
    for k in range(4)
]

batched_results = generate_img2img(
    pipe, image_proj, originals[:4], embeddings[:4], **shared_kwargs())

for k in range(4):
    assert imgs_equal(batched_results[k], b1_results[k]), (
        f'sample {k}: batched output differs from B=1, '
        f'max diff={max_abs_diff(batched_results[k], b1_results[k])}'
    )

print('Test 3 PASSED: B=4 distinct inputs each match their B=1 counterpart pixel-exactly')

## Test 4 — B=4 with per-sample `object_mask`

Build four object masks with different bbox radii (different `n_bbox` token
counts). Confirm the per-sample loop in `IPAFluxAttnProcessor` produces
results that match independent B=1 calls.

In [ ]:
# Four bboxes: circles of increasing radius centred at the image centre.
cx, cy = IMAGE_SIZE / 2, IMAGE_SIZE / 2
radii = [64, 96, 128, 160]   # pixels — each gives a different n_bbox count

masks_b1 = [
    build_object_region_mask(IMAGE_SIZE, cx, cy, r, device='cpu', dtype=torch.bfloat16)
    for r in radii
]  # each (1, 1024, 1)

obj_emb = embeddings[0]   # same object embedding for all

b1_mask_results = [
    generate_img2img(
        pipe, image_proj, originals[k], embeddings[k],
        object_siglip_embedding=obj_emb,
        object_mask=masks_b1[k],
        object_ip_scale=0.5,
        **shared_kwargs(),
    )
    for k in range(4)
]

masks_batched = torch.cat(masks_b1, dim=0)   # (4, 1024, 1)
obj_emb_batch = obj_emb.unsqueeze(0).expand(4, -1)   # (4, 1152)

batched_mask_results = generate_img2img(
    pipe, image_proj, originals[:4], embeddings[:4],
    object_siglip_embedding=obj_emb_batch,
    object_mask=masks_batched,
    object_ip_scale=0.5,
    **shared_kwargs(),
)

for k in range(4):
    assert imgs_equal(batched_mask_results[k], b1_mask_results[k]), (
        f'object_mask sample {k}: batched differs from B=1, '
        f'max diff={max_abs_diff(batched_mask_results[k], b1_mask_results[k])}'
    )

print('Test 4 PASSED: per-sample object_mask loop matches B=1 for all four bbox sizes')

## Test 5 — Guard clauses

B > 1 with `use_rf_inversion=True` must raise `NotImplementedError`.

In [ ]:
try:
    generate_img2img(
        pipe, image_proj, originals[:2], embeddings[:2],
        use_rf_inversion=True, **shared_kwargs())
    assert False, 'expected NotImplementedError'
except NotImplementedError:
    pass

print('Test 5 PASSED: use_rf_inversion=True with B > 1 raises NotImplementedError')

## Summary

In [ ]:
print('All tests passed.')